# Hydro Kaggle ATC Job
This notebook is generated automatically by Hydro to score a single submission with ATC.


In [ ]:
import base64
import glob
import json
import os
import shutil
import subprocess
import traceback
from datetime import datetime, timezone
from pathlib import Path

payload = json.loads("{\"code_b64\":\"I2luY2x1ZGUgPGlvc3RyZWFtPgojaW5jbHVkZSA8YWxnb3JpdGhtPgp1c2luZyBuYW1lc3BhY2Ugc3RkOwoKaW50IG1haW4oKSB7CiAgICBpbnQgQSwgQiwgQzsKICAgIGNpbiA+PiBBID4+IEIgPj4gQzsKCiAgICBpbnQgcmVzdWx0ID0gbWluKEEsIG1pbihCLCBDKSk7CiAgICBjb3V0IDw8IHJlc3VsdDsKCiAgICByZXR1cm4gMDsKfQ==\",\"language\":\"cpp\",\"method\":\"entropy\",\"base_model_name\":\"codellama/CodeLlama-7b-Instruct-hf\",\"infer_task\":true,\"prompt_style\":\"regular\",\"threshold\":-0.25,\"pattern_weight_mapping\":{\"comments\":0,\"docstrings\":0},\"dataset_source\":\"tranducngan/atcv1-source\",\"atc_project_dir\":\"ATC-main\",\"provider_name\":\"kaggle-atc\",\"result_filename\":\"ai-check-result.json\"}")
result_path = Path(f"/kaggle/working/{payload['result_filename']}")
try:
    payload = json.loads("{\"code_b64\":\"I2luY2x1ZGUgPGlvc3RyZWFtPgojaW5jbHVkZSA8YWxnb3JpdGhtPgp1c2luZyBuYW1lc3BhY2Ugc3RkOwoKaW50IG1haW4oKSB7CiAgICBpbnQgQSwgQiwgQzsKICAgIGNpbiA+PiBBID4+IEIgPj4gQzsKCiAgICBpbnQgcmVzdWx0ID0gbWluKEEsIG1pbihCLCBDKSk7CiAgICBjb3V0IDw8IHJlc3VsdDsKCiAgICByZXR1cm4gMDsKfQ==\",\"language\":\"cpp\",\"method\":\"entropy\",\"base_model_name\":\"codellama/CodeLlama-7b-Instruct-hf\",\"infer_task\":true,\"prompt_style\":\"regular\",\"threshold\":-0.25,\"pattern_weight_mapping\":{\"comments\":0,\"docstrings\":0},\"dataset_source\":\"tranducngan/atcv1-source\",\"atc_project_dir\":\"ATC-main\",\"provider_name\":\"kaggle-atc\",\"result_filename\":\"ai-check-result.json\"}")
    code = base64.b64decode(payload['code_b64']).decode('utf-8')

    project_dir = payload['atc_project_dir']
    dataset_owner, dataset_slug = (payload['dataset_source'].split('/', 1) + [''])[:2]
    search_patterns = ['/kaggle/input', '/kaggle/input/*', '/kaggle/input/*/*', '/kaggle/input/*/*/*', '/kaggle/input/*/*/*/*']
    candidate_dirs = []
    seen = set()
    for pattern in search_patterns:
        for candidate in glob.glob(pattern):
            candidate_path = Path(candidate)
            if not candidate_path.is_dir():
                continue
            candidate_path = candidate_path.resolve()
            key = str(candidate_path)
            if key in seen:
                continue
            seen.add(key)
            if (candidate_path / "requirements.txt").exists() and (candidate_path / "detection" / "run.py").exists():
                candidate_dirs.append(candidate_path)
    if not candidate_dirs:
        visible_inputs = [str(Path(path).resolve()) for path in glob.glob("/kaggle/input/*")]
        raise RuntimeError(f"Could not locate an ATC project under /kaggle/input. dataset_source={payload['dataset_source']!r}, visible_inputs={visible_inputs}")
    def candidate_score(candidate_path):
        candidate_str = str(candidate_path)
        score = 0
        if project_dir and candidate_path.name == project_dir:
            score += 8
        if dataset_slug and dataset_slug in candidate_str:
            score += 4
        if dataset_owner and dataset_owner in candidate_str:
            score += 2
        if "/datasets/" in candidate_str:
            score += 1
        return (-score, len(candidate_str), candidate_str)
    src = str(sorted(candidate_dirs, key=candidate_score)[0])
    dst = f'/kaggle/working/{project_dir}'
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    detector_file = Path(dst) / 'detection' / 'detector.py'
    detector_source = detector_file.read_text(encoding='utf-8')
    patched_detector_source = detector_source.replace('device_map="cuda"', 'device_map="auto"')
    if patched_detector_source != detector_source:
        detector_file.write_text(patched_detector_source, encoding='utf-8')
    subprocess.run(['pip', 'install', '-q', '-r', os.path.join(dst, 'requirements.txt')], check=True)
    os.chdir(dst)

    from detection.run import InferTaskConfig
    from detection.detector import EntropyDetector, MeanLogLikelihoodDetector, LogRankDetector, LRRDetector

    detectors = {
        'entropy': EntropyDetector,
        'mean_log_likelihood': MeanLogLikelihoodDetector,
        'log_rank': LogRankDetector,
        'lrr': LRRDetector,
    }
    detector_cls = detectors[payload['method']]
    infer_task_cfg = InferTaskConfig(debug=False, debug_file='debug_documentation.txt', use_cache=False, prompt_style=payload['prompt_style'])
    detector = detector_cls(model_name=payload['base_model_name'], pattern_weight_mapping=payload['pattern_weight_mapping'], infer_task_cfg=infer_task_cfg, language=payload['language'])
    inferred_task = None
    if payload['infer_task']:
        score, inferred_task = detector.compute_score_infer_task(code)
    else:
        score = detector.compute_score_without_task(code)
    threshold = float(payload['threshold'])
    is_ai = bool(score >= threshold)
    comparison = '>=' if is_ai else '<'
    message = f'Kaggle ATC score {score:.6f} {comparison} threshold {threshold:.6f}.'
    result = {
        'aiCheck': {
            'state': 'checked',
            'isAI': is_ai,
            'score': float(score),
            'provider': payload['provider_name'],
            'message': message,
            'checkedAt': datetime.now(timezone.utc).isoformat(),
        },
        'score': float(score),
        'threshold': threshold,
    }
    if inferred_task is not None:
        result['inferredTask'] = inferred_task
    result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(result, ensure_ascii=False))
except Exception as exc:
    error_result = {
        'aiCheck': {
            'state': 'error',
            'isAI': None,
            'score': None,
            'provider': payload['provider_name'],
            'message': f'{type(exc).__name__}: {exc}',
            'checkedAt': datetime.now(timezone.utc).isoformat(),
        },
        'traceback': traceback.format_exc(),
    }
    result_path.write_text(json.dumps(error_result, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(error_result, ensure_ascii=False))
